# Step 24: Fusion - the best model
Voice (eGeMAPS) + w2v-BERT 2.0 speech + VideoMAE video features together, with and without **past context** (mean of the last 5 s and 10 s). Two heads: Gradient Boosting and a dense network. Predictions are also **smoothed** over the last 5 s.
*Same as `src/24_fusion.py`. Slow (over an hour).*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page: https://multimediate-challenge.org/datasets/Dataset_NoXi/
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017, https://doi.org/10.1145/3136755.3136780
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"     # hide TensorFlow messages
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"    # same maths settings as the original script
if os.getcwd().endswith("notebooks"):
    os.chdir("..")                           # work from the project folder

In [ ]:
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingRegressor

In [ ]:
def ccc(y_true, y_pred):
    # Concordance Correlation Coefficient: 1 = perfect agreement
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    covariance = np.mean((y_true - y_true.mean()) * (y_pred - y_pred.mean()))
    return 2 * covariance / (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean()) ** 2 + 1e-9)

**Past mean:** for each row, the mean of the same speaker's last k seconds (only the past, so it would also work live). Used for **context** (features) and **smoothing** (predictions).

In [ ]:
def past_mean(X, speakers, k):
    # for every row: the mean of the same speaker's last k seconds (only the past)
    out = np.zeros(X.shape)
    for s in pd.unique(speakers):
        rows = np.where(speakers == s)[0]
        out[rows] = pd.DataFrame(X[rows]).rolling(k, min_periods=1).mean().to_numpy()
    return out

In [ ]:
df = pd.read_parquet("data/features/noxi_engagement.parquet")
voice = []
for c in df.columns:
    if c.startswith("voice_"):
        voice.append(c)
y = df["engagement"].to_numpy("float32")
speaker = (df["session"] + "_" + df["role"]).to_numpy()
language = df["language"].to_numpy()
session = df["session"].to_numpy()
languages = ["English", "French", "German"]

fused = np.hstack([df[voice].fillna(0).to_numpy("float32"),
                   np.load("data/features/pretrained_w2vbert2.npy"),
                   np.load("data/features/pretrained_videomae.npy")])
with_context = np.hstack([fused, past_mean(fused, speaker, 5).astype("float32"), past_mean(fused, speaker, 10).astype("float32")])
print(fused.shape[1], "features, with context:", with_context.shape[1])

In [ ]:
# the 3 sessions of each language
english = sorted(df[df["language"] == "English"]["session"].unique())
french = sorted(df[df["language"] == "French"]["session"].unique())
german = sorted(df[df["language"] == "German"]["session"].unique())

# fold 1 tests on the 1st session of each language, fold 2 on the 2nd, fold 3 on the 3rd
folds = []
for k in range(3):
    folds.append([english[k], french[k], german[k]])
print(folds)

**The two heads.**

In [ ]:
def gb_predict(X, train, test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X[train])
    X_test = scaler.transform(X[test])
    model = HistGradientBoostingRegressor(max_iter=300, random_state=42)
    model.fit(X_train, y[train])
    return model.predict(X_test)


def dense_predict(X, train, test):
    keras.utils.set_random_seed(42)
    model = keras.Sequential([
        layers.Input(shape=(X.shape[1],)),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.1),
        layers.Dense(1),
    ])
    model.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
    scaler = StandardScaler().fit(X[train])
    model.fit(np.clip(scaler.transform(X[train]), -5, 5), y[train], epochs=30, batch_size=128, verbose=0)
    return model.predict(np.clip(scaler.transform(X[test]), -5, 5), verbose=0).flatten()

**3 folds for every head and input** (raw and smoothed predictions).

In [ ]:
for head_name, predict in [("Gradient Boosting", gb_predict), ("dense head", dense_predict)]:
    for input_name, X in [("fusion, 1 s", fused), ("fusion + context", with_context)]:
        raw, smoothed = [], []
        for test_sessions in folds:
            test = np.isin(session, test_sessions)
            pred = predict(X, ~test, test)
            raw.append(ccc(y[test], pred))
            smooth_pred = past_mean(pred.reshape(-1, 1), speaker[test], 5).flatten()
            smoothed.append(ccc(y[test], smooth_pred))
        print(head_name.ljust(18), input_name.ljust(16), round(np.mean(raw), 3), "+-", round(np.std(raw, ddof=1), 3),
              "| smoothed", round(np.mean(smoothed), 3))

**Cross-language test of the best model** (Gradient Boosting, fusion + context).

In [ ]:
within = []
for l in languages:
    for s in sorted(df[df["language"] == l]["session"].unique()):
        within.append(ccc(y[session == s], gb_predict(with_context, (language == l) & (session != s), session == s)))
cross = []
for a in languages:
    for b in languages:
        if a != b:
            cross.append(ccc(y[language == b], gb_predict(with_context, language == a, language == b)))
drop = 100 * (np.mean(within) - np.mean(cross)) / np.mean(within)
print("within", round(np.mean(within), 3), "| cross", round(np.mean(cross), 3), "| drop", round(drop), "%")

**The result saved by the script**, for comparison.

In [ ]:
print(pd.read_csv("reports/fusion_results.csv")[["model", "input", "ccc", "smooth_ccc", "drop_pct"]].round(3))